In [ ]:
import pandas as pd
import re

# Load the initial datasets
file_paths = [
    "/mnt/data/Economic Data Release Dates Investing.com  - Jan 2024.csv",
    "/mnt/data/Economic Data Release Dates Investing.com  - Sheet19 (1).csv",
    "/mnt/data/Economic Data Release Dates Investing.com  - Sheet20 (1).csv",
    "/mnt/data/Economic Data Release Dates Investing.com  - Sheet16.csv",
    "/mnt/data/Economic Data Release Dates Investing.com  - Sheet17.csv",
    "/mnt/data/Economic Data Release Dates Investing.com  - Sheet18.csv"
]

# Read all files into dataframes and concatenate them into one dataframe
combined_df = pd.concat([pd.read_csv(file_path) for file_path in file_paths], ignore_index=True)

# Forward fill the missing dates
combined_df['Date'] = combined_df['Date'].ffill()

# Combine the 'Date' and 'Time' columns into a single 'DateTime' column and convert to pandas datetime format
combined_df['DateTime'] = pd.to_datetime(combined_df['Date'] + ' ' + combined_df['Time'], errors='coerce')

# Drop rows where DateTime could not be parsed
combined_df = combined_df.dropna(subset=['DateTime'])

# Sort the dataframe by DateTime in ascending order
combined_df_sorted = combined_df.sort_values(by='DateTime').reset_index(drop=True)

# Select relevant columns for the final dataset
final_combined_df = combined_df_sorted[['DateTime', 'Event']]

# Define the list of specific events we want to extract with exact matches ignoring parenthesis content
events_of_interest_exact = [
    "Atlanta Fed GDPNow",
    "Average Hourly Earnings",
    "CPI",
    "Core CPI",
    "Core CPI Index",
    "Core Durable Goods Orders",
    "Core PCE Price Index",
    "Core Retail Sales",
    "Durable Goods Orders",
    "Fed Interest Rate Decision",
    "ISM Manufacturing PMI",
    "ISM Non-Manufacturing PMI",
    "Nonfarm Payrolls",
    "PCE Price index",
    "Retail Sales",
    "Unemployment Rate",
    "FOMC Press Conference"
]

# Function to match events ignoring content in parenthesis
def match_event_with_parenthesis(event_name):
    event_name_cleaned = re.sub(r'\s*\(.*?\)\s*', '', event_name).strip()
    for target_event in events_of_interest_exact:
        if target_event.lower() == event_name_cleaned.lower():
            return True
    return False

# Filter the dataframe using the refined matching function
filtered_final_df_corrected = final_combined_df[final_combined_df['Event'].apply(match_event_with_parenthesis)]

# Select only the relevant columns for the final dataframe
result_final_df_corrected = filtered_final_df_corrected[['DateTime', 'Event']].dropna().reset_index(drop=True)

# Load the signal dataframe
signal_dataset_path = "/mnt/data/Signature_AI_Results_Final.csv"
signal_df = pd.read_csv(signal_dataset_path)

# Convert 'Datetime' column in the signal dataframe to pandas datetime format
signal_df['Datetime'] = pd.to_datetime(signal_df['Datetime'], errors='coerce')

# Merge the two dataframes on the date part of the datetime column
signal_df['Date'] = signal_df['Datetime'].dt.date
result_final_df_corrected['Date'] = result_final_df_corrected['DateTime'].dt.date

# Merge the two dataframes on the date column
merged_df_by_date_with_datetime = pd.merge(signal_df, result_final_df_corrected, left_on='Date', right_on='Date', how='left')

# Group by 'Datetime' in the merged dataframe and concatenate the events and event datetimes
merged_df_by_date_with_datetime['events'] = merged_df_by_date_with_datetime.groupby('Datetime')['Event'].transform(lambda x: ', '.join(x.dropna().unique()))
merged_df_by_date_with_datetime['event_datetimes'] = merged_df_by_date_with_datetime.groupby('Datetime')['DateTime'].transform(lambda x: ', '.join(pd.to_datetime(x.dropna()).dt.strftime('%Y-%m-%d %H:%M:%S').unique()))

# Drop the extra 'DateTime', 'Event', and 'Date' columns
merged_df_by_date_with_datetime = merged_df_by_date_with_datetime.drop(columns=['DateTime', 'Event', 'Date']).drop_duplicates()

# Save the updated signal dataframe to a CSV file
updated_signal_final_csv_path = "/mnt/data/Updated_Signal_Dataset_Final.csv"
merged_df_by_date_with_datetime.to_csv(updated_signal_final_csv_path, index=False)


In [ ]:
from datetime import timedelta

# Load the updated signal dataset
updated_signal_df = pd.read_csv(updated_signal_dataset_path)

# Convert 'Datetime' column to pandas datetime format
updated_signal_df['Datetime'] = pd.to_datetime(updated_signal_df['Datetime'], errors='coerce')


# Define a function to filter signals based on the time difference from event datetimes
def filter_signals(df, hours, before=True):
    if before:
        time_delta = -timedelta(hours=hours)
    else:
        time_delta = timedelta(hours=hours)
    filtered_indices = []
    for index, row in df.iterrows():
        event_datetimes = row['event_datetimes']
        if pd.notna(event_datetimes):
            event_times = [pd.to_datetime(time.strip()) for time in event_datetimes.split(',')]
            for event_time in event_times:
                if before and (row['Datetime'] >= event_time + time_delta) and (row['Datetime'] <= event_time):
                    filtered_indices.append(index)
                elif not before and (row['Datetime'] <= event_time + time_delta) and (row['Datetime'] >= event_time):
                    filtered_indices.append(index)
    filtered_df = df.drop(filtered_indices)
    return filtered_df

# Create and save the datasets for signals before events
before_datasets = {}
for hours in range(1, 6):
    before_datasets[hours] = filter_signals(updated_signal_df, hours, before=True)
    before_datasets[hours].to_csv(f"/mnt/data/Filtered_Signal_Before_{hours}_Hours.csv", index=False)

# Create and save the datasets for signals after events
after_datasets = {}
for hours in range(1, 6):
    after_datasets[hours] = filter_signals(updated_signal_df, hours, before=False)
    after_datasets[hours].to_csv(f"/mnt/data/Filtered_Signal_After_{hours}_Hours.csv", index=False)


before_csv_paths = [f"/mnt/data/Filtered_Signal_Before_{hours}_Hours.csv" for hours in range(1, 6)]
after_csv_paths = [f"/mnt/data/Filtered_Signal_After_{hours}_Hours.csv" for hours in range(1, 6)]

(before_csv_paths, after_csv_paths)

In [ ]:
# Define a function to filter signals within a certain time window before and after event datetimes
def filter_signals_before_after(df, hours):
    time_delta = timedelta(hours=hours)
    filtered_indices = []
    for index, row in df.iterrows():
        event_datetimes = row['event_datetimes']
        if pd.notna(event_datetimes):
            event_times = [pd.to_datetime(time.strip()) for time in event_datetimes.split(',')]
            for event_time in event_times:
                if (row['Datetime'] >= event_time - time_delta) and (row['Datetime'] <= event_time + time_delta):
                    filtered_indices.append(index)
    filtered_df = df.drop(filtered_indices)
    return filtered_df

# Create and save the datasets for signals within the specified hours before and after events
combined_datasets = {}
for hours in range(1, 6):
    combined_datasets[hours] = filter_signals_before_after(merged_df_by_date_with_datetime, hours)
    combined_datasets[hours].to_csv(f"/mnt/data/Filtered_Signal_Before_After_{hours}_Hours.csv", index=False)


combined_csv_paths = [f"/mnt/data/Filtered_Signal_Before_After_{hours}_Hours.csv" for hours in range(1, 6)]
combined_csv_paths